# LangChain RAG pipeline (modular)

Hadith RAG over `data/dorar_hadith_full_batch_2.csv`. Each section below is one pipeline stage; change **`CONFIG`** in the next cell to swap models, chunking, or retrieval without touching the rest.

**Run order:** Config → Imports → Load → Split → Embeddings → Vector store → Retriever → LLM → Prompt → Chain → Query.

**Requirements:** `pip install -r requirements.txt` (optional: copy `.env.example` to `.env` for API keys).

In [1]:
# import re
# import pandas as pd
# TASHKEEL = re.compile(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]')
#
# def normalize(text):
#     if pd.isna(text):
#         return text
#     text = re.sub(TASHKEEL, '', text)
#
#     # # normalize Arabic letters
#     # text = re.sub(r'[إأآ]', 'ا', text)
#     # text = re.sub(r'ى', 'ي', text)
#     # text = re.sub(r'ة', 'ه', text)
#     #
#     # # normalize prophet prayer forms
#     # text = re.sub(r'صل[ىي]\s+الله\s+عليه\s+وسلم', 'صلي الله عليه وسلم', text)
#     #
#     # text = re.sub(r'\s+', ' ', text).strip()
#
#     return text
#
# df = pd.read_csv("data/dorar_hadith_full_batch_2.csv")
# df['sharh'] = df['sharh'].apply(normalize)
# df.to_csv("data/dorar_hadith_without_tashkeel.csv")

In [23]:
from pathlib import Path

CONFIG = {
  "paths": {
    "data_csv": Path("data/dorar_hadith_without_tashkeel.csv"),
    "chroma_dir": Path("chroma_db"),
    "collection_name": "hadith_rag",
  },
  "data": {
    "max_rows": 500,          # None = full CSV (large). Start small while experimenting.
    "text_columns": [         # Columns merged into each document body
       "sharh",
    ],
    "metadata_columns":
        ["page_id", "url", "categories", "hadith_1",
         "rawy_1", "mohadth_1",
         "source_1", "hokm_1","categories",],
  },
  "chunking": {
    "chunk_size": 800,
    "chunk_overlap": 120,
    "separators": ["\n\n", "\n", ". ", " ", ""],
  },
  "embeddings": {
    "provider": "huggingface",  # "huggingface" | "openai"
    "model_name": "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2",
    "openai_model": "text-embedding-3-small",
  },
  "vector_store": {
    "persist": True,
    "reset_on_build": True,  # True = delete collection and re-index
  },
  "retriever": {
    "search_type": "similarity",  # "similarity" | "mmr"
    "k": 4,
    "fetch_k": 12,                # used when search_type == "mmr"
    "lambda_mult": 0.5,
  },
  "llm": {
    "provider": "qrok",       # "openai" | "ollama"
    "openai_model": "gpt-4o-mini",
    "ollama_model": "llama3.2",
    "groq_model"  :"llama-3.3-70b-versatile",
    "temperature" : 0.1,
  },
  "prompt": {
    "language": "ar",
    "system_role": (
      "انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمة"
      "اذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة"
    ),
  },
}

PROJECT_ROOT = Path(".").resolve()
CONFIG["paths"]["data_csv"] = PROJECT_ROOT / CONFIG["paths"]["data_csv"]
CONFIG["paths"]["chroma_dir"] = PROJECT_ROOT / CONFIG["paths"]["chroma_dir"]

In [24]:
import os
import shutil
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

def cfg(*keys: str) -> Any:
    """Read nested CONFIG values, e.g. cfg('retriever', 'k')."""
    node = CONFIG
    for key in keys:
        node = node[key]
    return node

In [25]:
def row_to_page_content(row: pd.Series) -> str:
    parts = []
    for col in cfg("data", "text_columns"): # for each column
        if col in row.index:
            val = str(row[col]).strip()
            if val and val.lower() != "nan":
                parts.append(f"{col}: {val}")
    return "\n".join(parts)

def load_hadith_documents() -> list[Document]:
    csv_path = cfg("paths", "data_csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    max_rows = cfg("data", "max_rows")
    if max_rows:
        df = df.head(max_rows)

    docs: list[Document] = []
    for _, row in df.iterrows():
        text = row_to_page_content(row)
        if not text.strip():
            continue
        metadata = {
            col: row[col] for col in cfg("data", "metadata_columns") if col in row.index and pd.notna(row[col])
        }
        docs.append(Document(page_content=text, metadata=metadata))

    print(f"Loaded {len(docs)} documents from {csv_path.name} ({len(df)} rows read)")
    return docs


raw_documents = load_hadith_documents()
raw_documents[0].page_content[:400] if raw_documents else "No documents"

Loaded 396 documents from dorar_hadith_without_tashkeel.csv (397 rows read)


'sharh: \ufeff صلى بنا النبي صلى الله عليه وسلم، فقام في الركعتين الأوليين قبل أن يجلس، فمضى في صلاته، فلما قضى صلاته انتظر الناس تسليمه، فكبر وسجد قبل أن يسلم، ثم رفع رأسه، ثم كبر وسجد، ثم رفع رأسه وسلم. الراوي : عبدالله بن مالك بن بحينة | المحدث : البخاري | المصدر : صحيح البخاري الصفحة أو الرقم: 6670 | خلاصة حكم المحدث : [صحيح] التخريج : أخرجه البيهقي (2841) واللفظ له، ومسلم (570)، وأبو داود (1034)، و'

In [26]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=cfg("chunking", "chunk_size"),
    chunk_overlap=cfg("chunking", "chunk_overlap"),
    separators=cfg("chunking", "separators"),
)

chunks = text_splitter.split_documents(raw_documents)
print(f"Split into {len(chunks)} chunks (avg ~{sum(len(c.page_content) for c in chunks) // max(len(chunks), 1)} chars)")
chunks[0].page_content[:300] if chunks else None

Split into 1670 chunks (avg ~560 chars)


'sharh: \ufeff صلى بنا النبي صلى الله عليه وسلم، فقام في الركعتين الأوليين قبل أن يجلس، فمضى في صلاته، فلما قضى صلاته انتظر الناس تسليمه، فكبر وسجد قبل أن يسلم، ثم رفع رأسه، ثم كبر وسجد، ثم رفع رأسه وسلم. الراوي : عبدالله بن مالك بن بحينة | المحدث : البخاري | المصدر : صحيح البخاري الصفحة أو الرقم: 6670 | '

In [27]:
def build_embeddings():
    provider = cfg("embeddings", "provider")
    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(model=cfg("embeddings", "openai_model"))
    if provider == "huggingface":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name=cfg("embeddings", "model_name"))
    raise ValueError(f"Unknown embeddings provider: {provider}")


embeddings = build_embeddings()
# Quick sanity check (optional; comment out on slow machines)
# len(embeddings.embed_query("اختبار"))
print(f"Embeddings ready: {cfg('embeddings', 'provider')} / {cfg('embeddings', 'model_name') if cfg('embeddings', 'provider') == 'huggingface' else cfg('embeddings', 'openai_model')}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6180.01it/s]


Embeddings ready: huggingface / Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2


In [7]:
from langchain_chroma import Chroma

chroma_dir = cfg("paths", "chroma_dir")
collection = cfg("paths", "collection_name")

if cfg("vector_store", "reset_on_build") and chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    print(f"Removed {chroma_dir}")

vectorstore = Chroma(
    collection_name=collection,
    embedding_function=embeddings,
    persist_directory=str(chroma_dir) if cfg("vector_store", "persist") else None,
)

def _collection_has_vectors(vs: Chroma) -> bool:
    data = vs.get(limit=1)
    return bool(data.get("ids"))


# Index only if collection is empty (re-run safe)
if not _collection_has_vectors(vectorstore):
    vectorstore.add_documents(chunks)
    print(f"Indexed {len(chunks)} chunks into '{collection}'")
else:
    n = len(vectorstore.get().get("ids", []))
    print(f"Using existing index: {n} vectors in '{collection}'")

vectorstore

Indexed 1670 chunks into 'hadith_rag'


In [28]:
# 3 -> 2135
# x -> 200,000
def build_retriever():
    search_type = cfg("retriever", "search_type")
    k = cfg("retriever", "k")

    if search_type == "similarity":
        return vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": k})
    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={
                "k": k,
                "fetch_k": cfg("retriever", "fetch_k"),
                "lambda_mult": cfg("retriever", "lambda_mult"),
            },
        )
    raise ValueError(f"Unknown search_type: {search_type}")


retriever = build_retriever()
print(f"Retriever: {cfg('retriever', 'search_type')}, k={cfg('retriever', 'k')}")

Retriever: similarity, k=4


In [29]:
def build_llm():
    provider = cfg("llm", "provider")
    temperature = cfg("llm", "temperature")

    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=cfg("llm", "openai_model"), temperature=temperature)
    if provider == "ollama":
        from langchain_community.chat_models import ChatOllama
        return ChatOllama(model=cfg("llm", "ollama_model"), temperature=temperature)
    if provider == "qrok":
        from langchain_groq import ChatGroq
        return ChatGroq(model=cfg("llm", "groq_model"), temperature=temperature)
        # max_tokens=1024
    raise ValueError(f"Unknown llm provider: {provider}")

llm = build_llm()
print(f"LLM: {cfg('llm', 'provider')}")

LLM: qrok


In [30]:
def format_docs(docs: list[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = ", ".join(f"{k}={v}" for k, v in doc.metadata.items())
        blocks.append(f"[{i}] ({meta})\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", cfg("prompt", "system_role")),
    (
        "human",
        "السياق المسترجع:\n{context}\n\n"
        "السؤال: {question}\n\n"
        f"أجب باللغة: {cfg('prompt', 'language')}. اذكر page_id عند الاقتباس إن وُجد.",
    ),
])

prompt = RAG_PROMPT
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n\nالسؤال: {question}\n\nأجب باللغة: ar. اذكر page_id عند الاقتباس إن وُجد.'), additional_kwargs={})])

In [31]:
# LCEL chain: question -> retrieve -> format -> prompt -> llm -> text
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Optional: inspect retrieval only (no LLM cost)
def retrieve(question: str, k: int | None = None):
    docs = retriever.invoke(question)
    if k:
        docs = docs[:k]
    return docs

rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000025EB8B72BA0>, search_kwargs={'k': 4})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='انت عالم دين اسلامي تجاوب علي اسئلة من خلال النص المسند اليك و حاول تجنب تاليف كلام ديني و قم بارفاق الاحاديث الواردة و صحتها و مصدر الاحاديث المتسخدمةاذا لم تجد جوابا في السياق . ارشده الي استشارة عالم اسلامي افضل للحصول علي اجابة دقيقة'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='السياق المسترجع:\n{context}\n\nالسؤال: {question}\n\nأجب باللغة: ar. اذكر page_id عند الاقتباس إن وُجد.'), addi

In [32]:
def ask(question: str, *, show_sources: bool = True) -> str:
    if show_sources:
        print("--- Retrieved chunks ---")
        for doc in retrieve(question):
            print(f"page_id={doc.metadata.get('page_id')} | {doc.page_content}... | {doc.metadata}")
        print("--- Answer ---")
    return rag_chain.invoke(question)

QUESTION = "ما حكم سجود السهو إذا زاد الإمام في الصلاة؟"
answer = ask(QUESTION)
print(answer)

--- Retrieved chunks ---
page_id=170048 | . وفي هذا الحديث يخبر عبد الله بن مالك ابن بحينة رضي الله عنه -وبحينة أم عبد الله- أن النبي صلى الله عليه وسلم صلى بهم إحدى الصلوات، وفي الصحيحين أنها كانت صلاة الظهر، فلما قام من السجود الثاني في الركعة الثانية، لم يجلس للتشهد الأوسط، وقام إلى الركعة الثالثة مباشرة، فمضى صلى الله عليه وسلم في الصلاة ولم يرجع إلى الجلوس واستكمل الركعتين الأخريين، فلما انتهى من الصلاة وتشهد التشهد الأخير، انتظر الناس أن يسلم وينهي الصلاة، ولكنه صلى الله عليه وسلم كبر وسجد للسهو سجدتين قبل أن يسلم، ثم رفع رأسه وسلم من الصلاة، ويشرع في سجدتي السهو ما يشرع في السجود عامة. وفي الحديث: مشروعية سجود السهو قبل التسليم. وفيه: وقوع السهو من الأنبياء عليهم الصلاة والسلام في الأفعال، وهذا غير مخل بمقام النبوة أو بشيء من الشريعة.... | {'rawy_1': 'عبدالله بن مسعود', 'url': 'https://dorar.net/hadith/sharh/170048', 'page_id': 170048, 'hokm_1': 'إسناده صحيح', 'categories': 'سهو - إذا صلى خمسا ، سهو - السهو في الفرض والتطوع ، سهو - تنبيه الإمام إذا سها ، سهو - سجود السهو بعد الت

## Customization cheat sheet

| Goal | Change in `CONFIG` |
|------|-------------------|
| Use full dataset | `"max_rows": None` |
| Rebuild vector DB | `"reset_on_build": True` (run vector-store cell once) |
| More context per answer | Increase `retriever.k` or `chunk_size` |
| Diverse retrieval | `"search_type": "mmr"` |
| Local LLM | `"llm": {"provider": "ollama", ...}` + run Ollama |
| OpenAI embeddings | `"embeddings": {"provider": "openai", ...}` |
| Different fields in chunks | Edit `data.text_columns` / `metadata_columns` |
| Swap only the prompt | Edit `prompt.system_role` or the `RAG_PROMPT` cell |

**Swap a component:** re-run from that cell downward (e.g. new embeddings → vector store → retriever → chain).